# Frozen-split model comparison

This notebook evaluates ResNet-50 first. After reviewing that run and available Kaggle compute, set `RUN_OTHER_MODELS=True` to add EfficientNet-B0 and MobileNetV3-Large. It reads the existing `final_reproducible_v1` manifest and verifies every split image SHA-256. It never generates or edits a split. The supplied ResNet-18 metrics are carried as a labeled, not-rerun reference.

Upload the project dataset tree (including `data/splits/final_reproducible_v1`) to Kaggle, then set `PROJECT_ROOT` below to its mounted location. Outputs go to `/kaggle/working/model_comparison_outputs`.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys
# Set this to the Kaggle input folder containing data/splits/final_reproducible_v1.
PROJECT_ROOT = Path('/kaggle/input/healthcare-document-processing')
OUTPUT_ROOT = Path('/kaggle/working/model_comparison_outputs')
SCRIPT_SOURCE = PROJECT_ROOT / 'scripts/run_model_comparison.py'
if not SCRIPT_SOURCE.exists(): raise FileNotFoundError(SCRIPT_SOURCE)
shutil.copy2(SCRIPT_SOURCE, OUTPUT_ROOT.parent / 'run_model_comparison.py')
RUN_OTHER_MODELS = False  # ResNet-50 first; enable after reviewing time/memory.
MODELS = ['resnet50'] + (['efficientnet_b0', 'mobilenet_v3_large'] if RUN_OTHER_MODELS else [])
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'not available')
print('Models:', MODELS)

In [ ]:
# Run the comparison runner from a writable working directory.
import os
os.environ['MODEL_COMPARISON_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['MODEL_COMPARISON_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
# In Kaggle, copy the script into /kaggle/working and run each requested model.
script = OUTPUT_ROOT.parent / 'run_model_comparison.py'
script.parent.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(script), '--models', *MODELS]
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)

## Artifacts and interpretation

The runner writes one directory per measured architecture with its checkpoint, predictions, epoch history, JSON/CSV classification report, confusion matrix CSV/PNG, and run metadata. A `model_comparison.csv` summarizes measured models and the provided ResNet-18 baseline. Test predictions are generated only after selecting the checkpoint by validation accuracy. Benchmark times are specific to the Kaggle session hardware and should not be generalized to other devices. Prescription support is small, so inspect its per-class recall and confusion counts directly.